## Research Notebook for PISA question and answers on different languages

## Libraries

In [54]:
import os, getpass

In [55]:
from dotenv import load_dotenv, find_dotenv

In [56]:
# from openai import OpenAI
from anthropic import Anthropic

In [57]:
load_dotenv(find_dotenv(usecwd=True))  # finds .env from current working dir upward

True

In [58]:
# --- 0) Setup: imports & config
import os, re, time, json, math, random
from typing import Dict, List, Tuple, Any, Optional
import pandas as pd
import time

In [59]:
from tqdm import tqdm

## Configuration

In [60]:
client = Anthropic()

In [61]:
MODEL_CLAUDE = "claude-opus-4-5-20251101"
# claude-opus-4-5-20251101
# claude-sonnet-4-5-20250929
# claude-haiku-4-5-20251001

In [62]:
SHEET_ID = "1QVPzB7uMwqJ6jCsHkwIILnXvDQIycpqkcV3bkiDpzyQ" 
WORKSHEET_NAME = "datasetE"  # change if needed "dataset"
RESULTS_CSV = "llm_eval_results_Claude.csv"
SAMPLE_PER_LANGUAGE = 1     # 5 per language
MAX_LANGUAGES = 2          # 43 languages total
SEED = 42

random.seed(SEED)

## Load data from Google Sheet

In [63]:
csv_url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet={WORKSHEET_NAME}"
try:
    df = pd.read_csv(csv_url)
except Exception as e:
    raise RuntimeError(
        "Failed to read the Google Sheet via CSV export. "
        "Make sure the sheet is shared as 'Anyone with the link can view', "
        f"ID is correct, and tab name matches. Underlying error: {e}"
    )

expected_cols = {
    "qid","language","question","context","options","gold",
    "answer_type","category","difficulty","rationale","source"
}
missing = expected_cols - set(df.columns)
if missing:
    raise ValueError(f"Your sheet is missing columns: {sorted(missing)}")

In [64]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   qid            8 non-null      object 
 1   language       8 non-null      object 
 2   language_code  8 non-null      object 
 3   question       8 non-null      object 
 4   context        8 non-null      object 
 5   options        8 non-null      object 
 6   gold           8 non-null      object 
 7   answer_type    8 non-null      object 
 8   category       8 non-null      object 
 9   difficulty     8 non-null      object 
 10  rationale      0 non-null      float64
 11  source         8 non-null      object 
dtypes: float64(1), object(11)
memory usage: 900.0+ bytes


## Normalize & sample

In [65]:
df["language"] = df["language"].astype(str).str.strip()
# pick first MAX_LANGUAGES languages (sorted)
languages = sorted(df["language"].unique())[:MAX_LANGUAGES]

In [66]:
# sample up to SAMPLE_PER_LANGUAGE per language
sampled = (
    df[df["language"].isin(languages)]
    .groupby("language", sort=True, group_keys=False)
    .head(SAMPLE_PER_LANGUAGE)
    .reset_index(drop=True)
)
if sampled.empty:
    raise ValueError("No rows selected. Check your data.")

In [67]:
print(f"Selected {len(sampled)} rows across {sampled['language'].nunique()} languages.")
display(sampled[["qid","language","question","gold"]])

Selected 2 rows across 2 languages.


,qid,language,question,gold
0,q023,German,Ist der Post relevant für das Problem von Inge...,A
1,q004,Indonesian,Perusahaan yang menyewakan truk memastikan bah...,C


## Parse options

In [68]:
def parse_options(raw: str) -> Dict[str, Any]:
    """
    Parse multiple-choice options from a JSON-encoded string.

    Supported formats
    -----------------
    1) Simple labeled strings (original format):
        ["A) 1575", "B) 8925", "C) 9000", "D) 9975"]

        -> {"A": "1575", "B": "8925", "C": "9000", "D": "9975"}

    2) List of dicts with labels mapping to lists of tokens:
        [
          {"A": ["India", "Colombia"]},
          {"B": ["India", "Armenia"]},
          {"C": ["Panama", "Colombia"]},
          {"D": ["Kazakhstan", "Colombia"]}
        ]
    """
    if pd.isna(raw):
        raise ValueError("Options are empty")

    # Load JSON
    try:
        items = json.loads(raw)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON format for options: {e}")

    if not isinstance(items, list):
        raise ValueError("Expected a JSON list at top level")

    # Case 1: list of strings -> original behavior
    if all(isinstance(item, str) for item in items):
        options: Dict[str, Any] = {}
        for item in items:
            # Match patterns like "A) text", "B. text", or "C: text"
            if not isinstance(item, str):
                raise ValueError(f"Option is not a string: {item}")
            match = re.match(r"^\s*([A-Z])[\)\.\:]\s*(.+)$", item.strip())
            if match:
                label, text = match.groups()
                options[label.upper()] = text.strip()
            else:
                # Fallback: assign next available letter automatically
                next_label = chr(ord('A') + len(options))
                options[next_label] = item.strip()
        return options

    # Case 2: list of dicts like [{"A": [...]}, {"B": [...]}]
    if all(isinstance(item, dict) for item in items):
        options: Dict[str, Any] = {}
        for idx, d in enumerate(items):
            if len(d) != 1:
                raise ValueError(
                    f"Each dict must have exactly one key (label). Problem at index {idx}: {d}"
                )
            (label_raw, value) = next(iter(d.items()))
            if not isinstance(label_raw, str):
                raise ValueError(f"Label must be a string, got: {label_raw}")

            label = label_raw.strip().upper()
            if not re.fullmatch(r"[A-Z]", label):
                raise ValueError(f"Invalid option label '{label_raw}' at index {idx}")

            # Accept list or scalar; normalize scalars into single-element lists if needed
            if isinstance(value, list):
                options[label] = value
            else:
                options[label] = [value]

        return options

    # If we reach here, the list is mixed or has unsupported types
    raise ValueError(
        "Unsupported options format: expected list of strings or list of single-key dicts"
    )


In [69]:
test = parse_options('["A) 2018", "B) 2019", "C) 2020", "D) 2021"]')
print(test)

{'A': '2018', 'B': '2019', 'C': '2020', 'D': '2021'}


In [70]:
options_block = "\n".join([f"{k}. {v}" for k,v in test.items()])
print(options_block)

A. 2018
B. 2019
C. 2020
D. 2021


## Build prompt

In [71]:
def build_prompt(row: pd.Series, options: Dict[str,str]) -> str:
    """
    Builds prompt for MCQ.
    """
    options_block = "\n".join([f"{k}. {v}" for k,v in options.items()])

    return (
        f"{row['context']}\n"
        f"{row['question']}\n"
        f"{options_block}\n\n"
        # f"{row['context']}\n"
        # f"{row['question']}\n"
        # f"{options_block}\n\n"
    )

In [72]:
answer_letter_regex = re.compile(r"\s*([A-Z])\s*[.)]?\s*")

def extract_letter(text: str, valid_letters: List[str]) -> str:
    """
    Extract the first single-letter A-Z token that is in valid_letters.
    """
    if not text:
        return ""
    # First line is the letter per our format; but still be defensive:
    first_line = text.splitlines()[0].strip().rstrip(".)").upper()
    # If first line is a single valid letter, use it
    if len(first_line) == 1 and first_line.upper() in valid_letters:
        return first_line.upper()
    # Else find any A-Z token
    m = answer_letter_regex.search(text.upper())
    if m and m.group(1) in valid_letters:
        return m.group(1)
    return ""

## LLM call wrapper

In [105]:
def llm_completion(
    prompt: str,
    model: Optional[str] = None,
    max_tokens: Optional[int] = None,
    system_prompt: str="Reply text inside the <reasoning> tags. \
        Output only the letter answer outside the tags.", #"Reply format: <LETTER>",
    retries: int = 2,
    backoff_seconds: float = 1.5,
    # reasoning_effort: Optional[str] = None,     # e.g., "low"|"medium"|"high" (and some models support "none")
    # reasoning_summary: Optional[str] = None,    # e.g., "auto" (or "concise"/"detailed" depending on model)
    **kwargs,
) -> Tuple[str, Dict[str, Any]]:
    """
    Call Anthropic Claude API and return (text, usage_dict).
    usage_dict contains: input_tokens, output_tokens, total_tokens (if present)
    """
    mdl = model or MODEL_CLAUDE
    last_err = None

    # Claude requires max_tokens
    if max_tokens is None:
        max_tokens = 4096

    effort_level = "high"

    for attempt in range(retries + 1):
        try:
            # Build arguments dynamically — include only if provided
            call_args = dict(
                model=mdl,
                betas=["effort-2025-11-24"],
                max_tokens=max_tokens,
                system=system_prompt,
                messages=[{"role": "user", "content": prompt}],
                output_config={
                    "effort": effort_level
                }
            )

            # Merge other kwargs (e.g., stop, seed, etc.)
            for k, v in kwargs.items():
                if k not in call_args:
                    call_args[k] = v

            # Actual model call
            response = client.beta.messages.create(**call_args)
            if response is None:
                return "No response from model.", {}
            
            # Extract text from response
            text = ""
            thinking_text = ""
            answer = ""
            
            for block in response.content:
                if block.type == "text":
                    text += block.text

            text = text.strip()

            def separate_thinking_and_answer(text: str) -> Tuple[str, str]:
                if not text:
                    return "", ""
                # Pattern to match <thinking>...</thinking> tags (with optional whitespace and newlines)
                thinking_pattern = r'<reasoning>(.*?)</reasoning>'
                # Extract all thinking content
                thinking_matches = re.findall(thinking_pattern, text, re.DOTALL | re.IGNORECASE)
                thinking_content = "\n\n".join(match.strip() for match in thinking_matches)
                # Remove all thinking tags and their content to get the final answer
                final_answer = re.sub(thinking_pattern, '', text, flags=re.DOTALL | re.IGNORECASE)
                final_answer = final_answer.strip()
                return thinking_content, final_answer
            thinking_text, answer = separate_thinking_and_answer(text)

            # --- Extract usage safely across SDK versions ---
            usage_obj = getattr(response, "usage", None)

            def _get(obj, key, default=None):
                if obj is None:
                    return default
                if isinstance(obj, dict):
                    return obj.get(key, default)
                return getattr(obj, key, default)

            usage: Dict[str, Any] = {
                "input_tokens": None,
                "output_tokens": None,
                "reasoning_tokens": None,
                "effort": effort_level,
                "summary": thinking_text,
            }

            if usage_obj is not None:
                # SDK objects often allow attribute access; dict-like in some contexts.
                
                usage["input_tokens"] = _get(usage_obj, "input_tokens")
                usage["output_tokens"] = _get(usage_obj, "output_tokens")

                out_details = _get(usage_obj, "output_tokens_details", {})
                usage["reasoning_tokens"] = _get(out_details, "reasoning_tokens")    
  
            return answer, thinking_text, usage

        except Exception as e:
            last_err = e
            if attempt < retries:
                time.sleep(backoff_seconds * (attempt + 1))
            else:
                raise

## Evaluation loop

In [106]:
def eval_rows(
        rows: pd.DataFrame, 
        cycle: int,
        model_tag: str,
        model_name: str, 
        results_path: str = RESULTS_CSV,
        sleep_s: float = 0.0, 
        retries: int = 2
    ) -> pd.DataFrame:
    results = []
    file_exists = os.path.exists(results_path)
    for i, row in tqdm(rows.iterrows(), total=len(rows), desc=f"Evaluating {model_tag} / {model_name}, cycle {cycle}"):
        qid = row["qid"]
        lang = row["language"]
        gold = str(row["gold"]).strip().upper()
        difficulty = row["difficulty"]

        # Parse options
        try:
            opts = parse_options(row["options"])
        except Exception as e:
            row_result = {
                "qid": qid,
                "language": lang,
                "pred": "",
                "gold": gold,
                "is_correct": False,
                "error": f"OptionsParseError: {e}",
                "raw": "",
                "difficulty": difficulty,
                "question": row["question"],
                "options_json": json.dumps(opts if 'opts' in locals() else {}, ensure_ascii=False),
                "model_tag": model_tag,
                "model_name": model_name,
                "cycle": cycle,
                "input_tokens": 0,
                "output_tokens": 0,
                "reasoning_tokens": 0,
                "effort": "Default",
                "summary": "",
            }
            results.append(row_result)

            # Save immediately
            pd.DataFrame([row_result]).to_csv(
                results_path,
                mode="a",
                header=not file_exists,
                index=False,
                encoding="utf-8"
            )
            file_exists = True
            continue

        valid_letters = sorted(list(opts.keys()))
        prompt = build_prompt(row, opts)

        # call model with simple retry
        raw = ""
        thinking = ""
        usage = {}
        err = ""
        for attempt in range(retries + 1):
            try:
                raw, thinking, usage = llm_completion(
                    prompt,
                    model=model_name,
                )
                break
            except Exception as e:
                err = f"{type(e).__name__}: {e}"
                if attempt < retries:
                    time.sleep(1.5 * (attempt + 1))
                else:
                    raw, thinking, usage = "", "", {}
        pred = extract_letter(raw, valid_letters)
        is_correct = (pred == gold)

        row_result = {
            "qid": qid,
            "language": lang,
            "pred": pred,
            "gold": gold,
            "is_correct": bool(is_correct),
            "error": err,
            "raw": raw,
            "difficulty": difficulty,
            "question": row["question"],
            "options_json": json.dumps(opts, ensure_ascii=False),
            "model_tag": model_tag,
            "model_name": model_name,
            "cycle": cycle,
            "input_tokens": usage.get("input_tokens"),
            "output_tokens": usage.get("output_tokens"),
            "reasoning_tokens": usage.get("reasoning_tokens"),
            "effort": usage.get("effort"),
            "summary": usage.get("summary", ""),
        }

        results.append(row_result)

        # *** Save this row immediately ***
        pd.DataFrame([row_result]).to_csv(
            results_path,
            mode="a",
            header=not file_exists,
            index=False,
            encoding="utf-8"
        )
        file_exists = True

        if sleep_s > 0:
            time.sleep(sleep_s)
    return pd.DataFrame(results)

## Execution

In [107]:
MODELS_TO_TEST = [
    ("Claude", MODEL_CLAUDE)
]

In [108]:
# filtered_df = df[df["language"] == "English"]
sampled = df
# sampled = filtered_df

In [109]:
N_CYCLES = 5  # repeat the same question to the same LLM

In [110]:
if os.path.exists(RESULTS_CSV):
    existing_results = pd.read_csv(RESULTS_CSV)
    print(f"Loaded existing results from {RESULTS_CSV}: {len(existing_results)} rows")
else:
    existing_results = pd.DataFrame()
    print("No existing results file found. Starting fresh.")

for tag, model_name in MODELS_TO_TEST:
    print(f"\nEvaluating {tag} -> {model_name}")
    for cycle in range(1, N_CYCLES + 1):
        # Determine which qids are already done for this (tag, model_name, cycle)
        if existing_results.empty:
            # Nothing done yet at all
            rows_to_eval = sampled.copy()
        else:
            # Subset only rows already done for this (tag, model_name, cycle)
            subset = existing_results[
                (existing_results["model_tag"] == tag) &
                (existing_results["model_name"] == model_name) &
                (existing_results["cycle"] == cycle)
            ]

            if subset.empty:
                # No rows done yet for this model+cycle
                rows_to_eval = sampled.copy()
            else:
                # Use (qid, language) pairs as the key — more robust than qid alone
                done_pairs = set(zip(subset["qid"], subset["language"]))

                mask = ~sampled.apply(
                    lambda r: (r["qid"], r["language"]) in done_pairs,
                    axis=1
                )
                rows_to_eval = sampled[mask]

        if rows_to_eval.empty:
            print(f"  • cycle {cycle}/{N_CYCLES}: already complete, skipping")
            continue

        print(f"  • cycle {cycle}/{N_CYCLES}: evaluating {len(rows_to_eval)} questions")
        t0 = time.time()

        df_new = eval_rows(
            rows_to_eval,
            model_tag=tag,
            model_name=model_name,
            cycle=cycle,
            results_path=RESULTS_CSV
        )

        elapsed = time.time() - t0
        print(f"    Done in {elapsed:.1f}s, newly evaluated {len(df_new)} rows.")

        # Update in-memory copy so subsequent cycles/ models can see freshly written rows
        existing_results = pd.concat([existing_results, df_new], ignore_index=True)

No existing results file found. Starting fresh.

Evaluating Claude -> claude-opus-4-5-20251101
  • cycle 1/5: evaluating 8 questions


Evaluating Claude / claude-opus-4-5-20251101, cycle 1: 100%|██████████| 8/8 [01:28<00:00, 11.06s/it]


    Done in 88.5s, newly evaluated 8 rows.
  • cycle 2/5: evaluating 8 questions


Evaluating Claude / claude-opus-4-5-20251101, cycle 2: 100%|██████████| 8/8 [01:22<00:00, 10.37s/it]


    Done in 83.0s, newly evaluated 8 rows.
  • cycle 3/5: evaluating 8 questions


...

    Done in 83.0s, newly evaluated 8 rows.
  • cycle 4/5: evaluating 8 questions


Evaluating Claude / claude-opus-4-5-20251101, cycle 4: 100%|██████████| 8/8 [01:23<00:00, 10.47s/it]


    Done in 83.8s, newly evaluated 8 rows.
  • cycle 5/5: evaluating 8 questions


Evaluating Claude / claude-opus-4-5-20251101, cycle 5: 100%|██████████| 8/8 [01:23<00:00, 10.47s/it]

    Done in 83.8s, newly evaluated 8 rows.


In [111]:
res_df = pd.read_csv(RESULTS_CSV) #RESULTS_CSV "llm_eval_results_Claude.csv"
print(f"\nTotal results loaded: {len(res_df)}")


Total results loaded: 40


## Results

In [112]:
overall_by_model = (
    res_df.groupby(["model_tag","model_name"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values("accuracy", ascending=False)
)
print("\nOverall accuracy by model:")
display(overall_by_model)


Overall accuracy by model:


,model_tag,model_name,accuracy
0,Claude,claude-opus-4-5-20251101,0.775


In [113]:
overall_by_question = (
    res_df.groupby(["qid"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values("accuracy", ascending=False)
)
print("\nOverall accuracy by question:")
display(overall_by_question)


Overall accuracy by question:


,qid,accuracy
1,q012,1.0
3,q018,1.0
2,q014,1.0
4,q023,0.6
0,q004,0.0


In [114]:
overall_by_lang = (
    res_df.groupby(["language"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values("accuracy", ascending=False)
)
print("\nOverall accuracy by lang:")
display(overall_by_lang)


Overall accuracy by lang:


,language,accuracy
3,Korean,1.0
2,Kazakh,1.0
4,Latvian,1.0
5,Russian,1.0
0,German,0.2
1,Indonesian,0.0


In [115]:
by_question = (
    res_df.groupby(["model_tag","model_name","language","qid"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values(["accuracy"])
)

# print("\nAccuracy by question:")
# display(by_question)

accuracy_counts_total = (
    by_question["accuracy"]
    .value_counts()
    .rename_axis("accuracy")
    .reset_index(name="count")
    .sort_values("accuracy")
)

print("\nCount of total questions by accuracy:")
display(accuracy_counts_total)

accuracy_counts = (
    by_question
    .groupby(["model_tag", "model_name", "accuracy"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .sort_values(["model_tag", "model_name", "accuracy"])
)

print("\nCount of questions by accuracy:")
display(accuracy_counts)


Count of total questions by accuracy:


,accuracy,count
1,0.0,1
2,0.2,1
0,1.0,6



Count of questions by accuracy:


,model_tag,model_name,accuracy,count
0,Claude,claude-opus-4-5-20251101,0.0,1
1,Claude,claude-opus-4-5-20251101,0.2,1
2,Claude,claude-opus-4-5-20251101,1.0,6


In [116]:
def safe_acc(s):
    return float('nan') if s.empty else s.mean()
overall_acc = safe_acc(res_df["is_correct"])
print(f"\nCombined overall accuracy on {len(res_df)} items: {overall_acc:.3f}")


Combined overall accuracy on 40 items: 0.775


In [117]:
for tag, _ in MODELS_TO_TEST:
    out_path = f"llm_eval_results__{tag}.csv"
    res_df.query("model_tag == @tag").to_csv(out_path, index=False)
    print(f"Saved {tag} results to: {out_path}")

# (Optional) quick peek
display(res_df.head())

Saved Claude results to: llm_eval_results__Claude.csv


,qid,language,pred,gold,is_correct,error,raw,difficulty,question,options_json,model_tag,model_name,cycle,input_tokens,output_tokens,reasoning_tokens,effort,summary
0,q023,German,A,A,True,NaN,A,level 2,Ist der Post relevant für das Problem von Inge...,"{""A"": [""Ja"", ""Ja"", ""Nein"", ""Nein"", ""Ja""], ""B"":...",Claude,claude-opus-4-5-20251101,1,920,588,NaN,high,Inge_88's problem is: She wants to know if it'...
1,q004,Indonesian,D,C,False,NaN,"D. Dia tidak benar, karena tidak satupun dari ...",level 6,Perusahaan yang menyewakan truk memastikan bah...,"{""A"": ""Manakah satu di antara pernyataan berik...",Claude,claude-opus-4-5-20251101,1,549,810,NaN,high,Mari saya analisis masalah ini.\n\n**Dimensi T...
2,q018,Kazakh,A,A,True,NaN,A,level 5,Пайымдау факті ме немесе пікір ме?\nСүттің ада...,"{""A"": [""Пікір"", ""Факт"", ""Факт"", ""Пікір""], ""B"":...",Claude,claude-opus-4-5-20251101,1,2768,643,NaN,high,"Бұл сұрақта төрт пайымдаудың факт пе, әлде пік..."
3,q012,Korean,B,B,True,NaN,B,level 5,이 기사에서 과학자들이 언급한 의견과 제레드 다이아몬드의 의견이 일치하는 부분은 무...,"{""A"": ""수백 년 전에 사람들이 라파누이에 정착했다."", ""B"": ""라파누이에서...",Claude,claude-opus-4-5-20251101,1,926,795,NaN,high,이 문제는 기사에서 언급된 과학자들의 의견과 제레드 다이아몬드의 의견이 일치하는 부...
4,q014,Korean,B,B,True,NaN,B,level 4,"블로그에 의하면, 이 교수가 현장 연구를 시작한 것은 언제인가?","{""A"": ""1990년대"", ""B"": ""9개월 전"", ""C"": ""1년 전"", ""D""...",Claude,claude-opus-4-5-20251101,1,1212,178,NaN,high,"블로그 글을 분석해보면:\n\n- 글이 올라온 날짜: 5월 23일\n- ""이번 주가..."


In [118]:
df_false = res_df[res_df['is_correct'] == False]
display(df_false.head(50))

,qid,language,pred,gold,is_correct,error,raw,difficulty,question,options_json,model_tag,model_name,cycle,input_tokens,output_tokens,reasoning_tokens,effort,summary
1,q004,Indonesian,D,C,False,NaN,"D. Dia tidak benar, karena tidak satupun dari ...",level 6,Perusahaan yang menyewakan truk memastikan bah...,"{""A"": ""Manakah satu di antara pernyataan berik...",Claude,claude-opus-4-5-20251101,1,549,810,NaN,high,Mari saya analisis masalah ini.\n\n**Dimensi T...
8,q023,German,C,A,False,NaN,C,level 2,Ist der Post relevant für das Problem von Inge...,"{""A"": [""Ja"", ""Ja"", ""Nein"", ""Nein"", ""Ja""], ""B"":...",Claude,claude-opus-4-5-20251101,2,920,567,NaN,high,"Inge_88 fragt, ob sie ihrer verletzten Henne A..."
9,q004,Indonesian,D,C,False,NaN,D,level 6,Perusahaan yang menyewakan truk memastikan bah...,"{""A"": ""Manakah satu di antara pernyataan berik...",Claude,claude-opus-4-5-20251101,2,549,726,NaN,high,Mari saya analisis masalah ini:\n\n**Dimensi T...
16,q023,German,C,A,False,NaN,C,level 2,Ist der Post relevant für das Problem von Inge...,"{""A"": [""Ja"", ""Ja"", ""Nein"", ""Nein"", ""Ja""], ""B"":...",Claude,claude-opus-4-5-20251101,3,920,603,NaN,high,Inge_88's problem: She wants to know if it's o...
17,q004,Indonesian,D,C,False,NaN,"D. Dia tidak benar, karena tidak satupun dari ...",level 6,Perusahaan yang menyewakan truk memastikan bah...,"{""A"": ""Manakah satu di antara pernyataan berik...",Claude,claude-opus-4-5-20251101,3,549,763,NaN,high,Mari saya analisis masalah ini.\n\n**Dimensi T...
24,q023,German,C,A,False,NaN,C,level 2,Ist der Post relevant für das Problem von Inge...,"{""A"": [""Ja"", ""Ja"", ""Nein"", ""Nein"", ""Ja""], ""B"":...",Claude,claude-opus-4-5-20251101,4,920,604,NaN,high,Inge_88's problem: She wants to know if it's o...
25,q004,Indonesian,D,C,False,NaN,D,level 6,Perusahaan yang menyewakan truk memastikan bah...,"{""A"": ""Manakah satu di antara pernyataan berik...",Claude,claude-opus-4-5-20251101,4,549,777,NaN,high,Mari saya analisis masalah ini.\n\n**Data Truk...
32,q023,German,C,A,False,NaN,C,level 2,Ist der Post relevant für das Problem von Inge...,"{""A"": [""Ja"", ""Ja"", ""Nein"", ""Nein"", ""Ja""], ""B"":...",Claude,claude-opus-4-5-20251101,5,920,576,NaN,high,Let me analyze each post for relevance to Inge...
33,q004,Indonesian,D,C,False,NaN,D,level 6,Perusahaan yang menyewakan truk memastikan bah...,"{""A"": ""Manakah satu di antara pernyataan berik...",Claude,claude-opus-4-5-20251101,5,549,749,NaN,high,Mari kita analisis masalah ini langkah demi la...


In [119]:
df_summary = (
    df_false
    .groupby(['qid'])
    .size()
    .reset_index(name='num_incorrect')
)

print(df_summary)


    qid  num_incorrect
0  q004              5
1  q023              4


In [120]:
df_summary = (
    df_false
    .groupby(['cycle'])
    .size()
    .reset_index(name='num_incorrect')
)

print(df_summary)


   cycle  num_incorrect
0      1              1
1      2              2
2      3              2
3      4              2
4      5              2
